# Model C_6: Salience weighting (bobot per kalimat berdasarkan isi berita)

**Ide.** Tidak semua kalimat berita sama penting untuk kurs. *"Strait of Hormuz blocked"* harus berbobot
jauh lebih besar daripada berita yang tidak berhubungan dengan dolar. Bobot dihitung oleh **scorer beku**
(model kemiripan kalimat yang TIDAK pernah melihat target kurs), lalu dipakai untuk merangkum kalimat menjadi
vektor harian.

Fokus eksperimen ini adalah **isi** (berita *apa* yang penting), bukan **waktu** (poin 1b.v). Agregasi
waktu (jendela 20 hari) dibuat sama untuk semua varian supaya perbedaan skor hanya berasal dari pembobotan isi.

**Pipeline**
1. Kalimat: judul + lead per artikel (sama seperti `05b`, C2 "art"), lead dipecah per kalimat.
2. Scorer beku (MiniLM) memberi tiap kalimat:
   - skor per **channel** transmisi kurs (minyak, Fed, risk-off, China, Indonesia/EM, komoditas) = kemiripan
     maksimum ke kalimat-kalimat query channel itu, distandardisasi dengan statistik train lalu sigmoid;
   - **relevansi** r(s) = skor channel tertinggi × (1 − skor channel *noise*);
   - **novelty** n(s) = 1 − kemiripan maksimum ke kalimat k hari trading sebelumnya.
3. Bobot w(s) = r(s) × n(s); bagian per channel w_c(s) = w(s) × softmax(z_c / τ).
4. Pooling hierarkis: kalimat → artikel (rata-rata berbobot), artikel → hari (bobot artikel = bobot kalimat terkuatnya).
5. Fitur harian: vektor FinBERT berbobot (PCA 20) + intensitas channel + ringkasan novelty + volume.
6. Jendela waktu t−19..t (baris tanggal t memprediksi arah kurs hari berikutnya, jadi sama dengan
   "hari t−20..t−1 → hari t"): (a) rata-rata *decay* eksponensial + HGB/LR, (b) GRU kecil.

**Ablasi** (agregasi waktu sama untuk semua baris):

| Varian | Bobot isi | Fitur channel | Novelty | Pooling |
|---|---|---|---|---|
| `base` | sama rata | – | – | flat |
| `rel` | relevansi | – | – | flat |
| `chan` | relevansi | ✓ | – | flat |
| `nov` | relevansi × novelty | ✓ | ✓ | flat |
| `hier` | relevansi × novelty | ✓ | ✓ | kalimat → artikel → hari |

**Aturan.** Query channel boleh diubah sesering apa pun selama hanya melihat bagian *Cek kewajaran* (data
train, tanpa target). Setelah melihat skor validation, bekukan. Mengubahnya sesudah itu = tuning di validation.

Jalankan dari folder `src/`. Butuh `torch`, `transformers`, `sentence-transformers`. Embedding disimpan di
`data/features_c/` jadi hanya dihitung sekali (GPU disarankan, CPU juga bisa tapi lebih lama).

In [ ]:
from c_common import *
import c_common
import os, re, time
from functools import lru_cache
from sklearn.decomposition import PCA
import torch

D, text = load_all()                      # corpus B_2, sama dengan C_2
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"
DEV = "cuda" if torch.cuda.is_available() else "cpu"

#Mode smoke test (C6_SMOKE=1): encoder diganti TF-IDF+SVD murah supaya pipeline bisa diuji tanpa GPU/internet.
SMOKE = os.environ.get("C6_SMOKE") == "1"
if SMOKE:
    import tempfile
    _tmp = Path(tempfile.mkdtemp(prefix="c6_smoke_"))
    c_common.RESULTS_C = _tmp / "results_c"
    CACHE = _tmp
    print("SMOKE TEST: encoder palsu, hasil ke", _tmp)
else:
    CACHE = FEATURES_C
CACHE.mkdir(parents=True, exist_ok=True)
print({s: len(d) for s, d in D.items()}, text.shape, "| device:", DEV)

## 1. Pengaturan

`CHANNELS` dan `NOISE` boleh kamu ubah: tambah/hapus channel, ganti kalimat, ubah jumlahnya. Tips:
- tulis seperti **judul berita**, bukan kata kunci;
- masukkan **dua arah** (naik dan turun) supaya relevansi tidak bergantung arah;
- jumlah kalimat per channel sebaiknya mirip (3–8). Kalibrasi per channel meredam efek jumlah, tapi tidak sempurna;
- hindari dua channel yang isinya hampir sama (fiturnya jadi kembar);
- tiap channel tambahan = 2 fitur baru (massa dan maksimum).

In [ ]:
SCORER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   
REPR_MODEL = "ProsusAI/finbert"                       
HF_CACHE = None              
BATCH = 32                
MAX_LEN = 128                #token per kalimat untuk FinBERT (kalimat judul/lead pendek)

WINDOW = 20                  
N_COMP = 20                  #PCA vektor harian (sama dengan 1a)
R_SHIFT = 1.0               

GRID_TAU = [0.5, 1.0]  
GRID_K = [3, 5]              
GRID_HALF = [2, 5, 10] 

CHANNELS = {
    "oil_energy": [
        "OPEC+ agrees to cut oil production",
        "Oil prices surge as supply fears grow",
        "Crude oil prices plunge on weak demand",
        "Shipping through the Strait of Hormuz is disrupted",
        "Sanctions hit Russian oil and gas exports",
    ],
    "fed_us_rates": [
        "The Federal Reserve raises interest rates",
        "Fed signals interest rate cuts as inflation cools",
        "US inflation comes in hotter than expected",
        "Treasury yields jump after a strong jobs report",
        "Powell says the Fed will keep rates higher for longer",
    ],
    "risk_off_conflict": [
        "Military strikes escalate the conflict in the Middle East",
        "Russia launches a new offensive in Ukraine",
        "Global stocks tumble as investors flee to safe havens",
        "Tensions rise over Taiwan as China holds military drills",
        "Ceasefire agreement eases fears of a wider war",
    ],
    "china_trade": [
        "The US imposes new tariffs on Chinese imports",
        "China retaliates in the escalating trade war",
        "China's economy slows as factory activity contracts",
        "Beijing announces stimulus to support growth",
        "Washington tightens export controls on chips to China",
    ],
    "indonesia_em": [
        "Bank Indonesia holds its benchmark interest rate",
        "The rupiah weakens against the US dollar",
        "Foreign investors pull money out of emerging markets",
        "Indonesia's economy grows faster than expected",
        "Emerging market currencies slide as the dollar strengthens",
    ],
    "commodities": [
        "Coal prices rise on strong demand from Asia",
        "Palm oil prices fall amid rising stockpiles",
        "Nickel prices jump after export restrictions",
        "Commodity prices rally on hopes of Chinese demand",
        "Gold hits a record high as investors seek safety",
    ],
}

#Hal yang tidak relevan untuk kurs. Kalimat yang mirip ini diturunkan bobotnya.
NOISE = [
    "The team won the championship game",
    "A celebrity announces a new movie role",
    "Tips for a healthy diet and better sleep",
    "Local police investigate a shooting",
    "A new smartphone is released with a better camera",
    "Lawmakers debate changes to Social Security benefits",
]
CH = list(CHANNELS)

## 2. Tabel kalimat

Artikel dipecah persis seperti `05b`/`05e` (judul dan lead dipisah `" | "`; kalau jumlahnya tidak cocok,
dipakai judul saja). Karena urutannya sama, `art` di sini = `id` di label C5 (`c5_llm_labels.jsonl`).
Tiap artikel → kalimat ke-0 = judul, lalu kalimat-kalimat lead (minimal 3 kata, duplikat dibuang).

In [ ]:
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'“])")


def build_sentence_table(text_df):
    rows, art_id, fallback = [], 0, 0
    for pos, (t, l) in enumerate(zip(text_df[TITLE].fillna(""), text_df[LEAD].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        if not titles:
            continue
        leads = [x.strip() for x in l.split(" | ")]
        if len(leads) != len(titles):          #judul saja
            leads, fallback = [""] * len(titles), fallback + 1
        for ti, li in zip(titles, leads):
            sents, seen = [], set()
            for s in [ti] + [x.strip() for x in SENT_SPLIT.split(li)]:
                key = s.lower().strip(" .")
                if len(s.split()) >= 3 and key not in seen:
                    seen.add(key)
                    sents.append(s)
            if not sents:                      
                sents = [ti]
            rows += [(pos, art_id, j, s) for j, s in enumerate(sents)]
            art_id += 1
    S = pd.DataFrame(rows, columns=["day", "art", "sent_no", "sentence"])
    S["date"] = text_df.index[S["day"].values]
    split_of = {d: s for s in D for d in D[s]["date"]}
    S["split"] = S["date"].map(split_of).fillna("warmup")   
    print(f"{art_id} artikel, {len(S)} kalimat, {S.groupby('art').size().mean():.2f} kalimat/artikel | "
          f"hari judul saja: {fallback}")
    return S


S = build_sentence_table(text)
S.groupby("split").size()

## 3. Encoding (sekali, lalu dari cache)

- **Scorer** (MiniLM, vektor dinormalisasi): untuk skor channel dan novelty. Tidak dipakai sebagai fitur.
- **Representasi** (FinBERT [CLS], sama dengan varian terbaik 1a): vektor yang dirangkum dengan bobot.

In [ ]:
_models = {}


def encode_scorer(texts):
    if SMOKE:
        return _smoke_encode(texts, 64)
    if "scorer" not in _models:
        from sentence_transformers import SentenceTransformer
        _models["scorer"] = SentenceTransformer(SCORER_MODEL, cache_folder=HF_CACHE, device=DEV)
    return _models["scorer"].encode(list(texts), batch_size=128, normalize_embeddings=True,
                                    show_progress_bar=len(texts) > 1000).astype("float32")


@torch.no_grad()
def encode_finbert(texts):
    if SMOKE:
        return _smoke_encode(texts, 48)
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(REPR_MODEL, cache_dir=HF_CACHE)
    model = AutoModel.from_pretrained(REPR_MODEL, cache_dir=HF_CACHE).to(DEV).eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])           # urut panjang -> padding minimal
    out = np.zeros((len(texts), model.config.hidden_size), dtype="float32")
    for s in range(0, len(order), BATCH):
        ix = order[s:s + BATCH]
        enc = tok([texts[i] for i in ix], padding=True, truncation=True, max_length=MAX_LEN,
                  return_tensors="pt").to(DEV)
        out[ix] = model(**enc).last_hidden_state[:, 0].float().cpu().numpy()
        if (s // BATCH) % 300 == 0:
            print(f"  FinBERT {s}/{len(texts)}")
    model.cpu()
    return out


def _smoke_encode(texts, dim):
    #hanya untuk smoke test: TF-IDF + SVD di-fit pada kalimat korpus.
    if "smoke" not in _models:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        vec = TfidfVectorizer(min_df=2, ngram_range=(1, 2), max_features=50000, sublinear_tf=True)
        X = vec.fit_transform(S["sentence"])
        _models["smoke"] = (vec, TruncatedSVD(64, random_state=SEED).fit(X))
    vec, svd = _models["smoke"]
    Z = svd.transform(vec.transform(list(texts)))[:, :dim]
    return (Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)).astype("float32")


def cached(name, fn):
    f = CACHE / name
    if f.exists():
        arr = np.load(f)
        if len(arr) == len(S):
            print("dimuat dari cache:", f.name, arr.shape)
            return arr
        print(f.name, "tidak cocok dengan jumlah kalimat -> hitung ulang")
    t0 = time.time()
    arr = fn(S["sentence"].tolist())
    np.save(f, arr)
    print(f"disimpan {f.name} {arr.shape} ({time.time() - t0:.0f} dtk)")
    return arr


E_sc = cached("c6_scorer_minilm.npy", encode_scorer)      
E_rep = cached("c6_finbert_sent_cls.npy", encode_finbert) 

## 4. Skor channel, relevansi, novelty

Untuk tiap kalimat dan tiap channel: kemiripan kosinus ke semua query channel itu, ambil **maksimum**
(sub-topik satu channel bisa berbeda jauh, misalnya "OPEC cut" vs "Hormuz blocked"). Skor mentah lalu
distandardisasi per channel memakai mean/std **kalimat train saja** (z), kemudian `sigmoid(z − R_SHIFT)`.

In [ ]:
q_texts = [q for c in CH for q in CHANNELS[c]] + NOISE
q_owner = np.array([c for c in CH for _ in CHANNELS[c]] + ["noise"] * len(NOISE))
Q = encode_scorer(q_texts)

sim = E_sc @ Q.T                           
RAW = np.column_stack([sim[:, q_owner == c].max(1) for c in CH + ["noise"]])
is_tr = (S["split"] == "train").values
mu, sd = RAW[is_tr].mean(0), RAW[is_tr].std(0) + 1e-8       
Z = (RAW - mu) / sd
P = 1 / (1 + np.exp(-(Z - R_SHIFT)))
Z_ch, P_ch, P_noise = Z[:, :len(CH)], P[:, :len(CH)], P[:, -1]
S["relevance"] = P_ch.max(1) * (1 - P_noise)
S["top_channel"] = np.array(CH)[P_ch.argmax(1)]


def channel_split(tau):
    #pembagian bobot kalimat ke channel 
    e = np.exp((Z_ch - Z_ch.max(1, keepdims=True)) / tau)
    return e / e.sum(1, keepdims=True)


def novelty(E, day, k):
    #1 - kemiripan maksimum ke kalimat dari k hari trading sebelumnya
    nov = np.ones(len(E), dtype="float32")
    by_day = {d: np.where(day == d)[0] for d in np.unique(day)}
    for d, ix in by_day.items():
        prev = [by_day[p] for p in range(d - k, d) if p in by_day]
        if prev:
            pv = np.concatenate(prev)
            nov[ix] = 1 - (E[ix] @ E[pv].T).max(1)
    return np.clip(nov, 0, 1)


NOV = {k: novelty(E_sc, S["day"].values, k) for k in GRID_K}

print("skor mentah (train): mean / std per channel")
display(pd.DataFrame({"mean": mu, "std": sd}, index=CH + ["noise"]).round(3).T)
print("relevansi (train):", S.loc[is_tr, "relevance"].describe().round(3).to_dict())
print({k: round(float(v[is_tr].mean()), 3) for k, v in NOV.items()}, "<- rata-rata novelty train per k")

## 5. Cek kewajaran (train saja, tanpa target)

Di sinilah query channel boleh disetel. Yang dicek:
1. kalimat dengan skor tertinggi per channel harus memang tentang channel itu;
2. kalimat dengan relevansi terendah harus memang tidak penting untuk kurs;
3. kecocokan dengan label event dari Qwen (C5): artikel berlabel "oil/energy" seharusnya skor `oil_energy`-nya tinggi, artikel "other" seharusnya relevansinya rendah.

In [ ]:
pd.set_option("display.max_colwidth", 120)
tr_S = S[is_tr]
for j, c in enumerate(CH):
    sc = P_ch[is_tr, j] * (1 - P_noise[is_tr])
    top = tr_S.assign(score=sc).nlargest(6, "score")[["date", "score", "sentence"]]
    print(f"\n=== {c}: skor tertinggi")
    display(top.round(3))
print("\n=== relevansi TERENDAH (seharusnya tidak penting)")
display(tr_S.nsmallest(8, "relevance")[["date", "relevance", "sentence"]].round(3))

In [ ]:
f_c5 = FEATURES_C / "c5_llm_labels.jsonl"
if f_c5.exists():
    c5 = pd.read_json(f_c5, lines=True)
    c5 = c5[c5["valid"]].set_index("id")
    art = S[is_tr].copy()
    for j, c in enumerate(CH):
        art[c] = P_ch[is_tr, j]
    art = art.groupby("art")[CH + ["relevance"]].max()  
    art = art.join(c5[["event", "region"]], how="inner")
    print(len(art), "artikel train dengan label C5")
    print("\nRata-rata skor artikel per label EVENT C5 (baris = label Qwen, kolom = channel kita):")
    display(art.groupby("event")[CH + ["relevance"]].mean().round(3).style.background_gradient(axis=0))
    print("\nPer label REGION C5:")
    display(art.groupby("region")[CH + ["relevance"]].mean().round(3).style.background_gradient(axis=0))
else:
    print("c5_llm_labels.jsonl tidak ada, cek C5 dilewati")

In [ ]:
k0 = GRID_K[0]
tmp = S[is_tr].assign(w=S["relevance"][is_tr] * NOV[k0][is_tr])
for d in tmp.groupby("date")["w"].sum().nlargest(5).index:
    g = tmp[tmp["date"] == d].nlargest(3, "w")
    print(f"\n{d.date()}  (total bobot {tmp.loc[tmp['date'] == d, 'w'].sum():.2f})")
    for _, r in g.iterrows():
        print(f"   w={r.w:.2f}  [{r.top_channel}]  {r.sentence[:110]}")

## 6. Fitur harian per varian

- vektor harian: rata-rata berbobot vektor FinBERT (flat, atau hierarkis untuk `hier`), lalu PCA 20 (fit pada hari train yang punya berita);
- channel (`chan`/`nov`/`hier`): `log1p(Σ w_c)` dan `max w_c` per channel;
- novelty (`nov`/`hier`): rata-rata dan maksimum novelty kalimat, porsi artikel "cerita baru" (novelty artikel di atas median train);
- volume: `has_news`, `log1p(jumlah artikel)`.

In [ ]:
VARIANTS = ["base", "rel", "chan", "nov", "hier"]
USES = {  
    "base": (False, False, False, False),
    "rel":  (True,  False, False, False),
    "chan": (True,  True,  False, False),
    "nov":  (True,  True,  True,  False),
    "hier": (True,  True,  True,  True),
}
N_DAYS = len(text)
DAY = S["day"].values
ART = S["art"].values
tr_dates = D["train"]["date"]


def group_wmean(keys, vecs, w):
    num = pd.DataFrame(vecs * w[:, None]).groupby(keys).sum()
    return num.div(pd.Series(w).groupby(keys).sum(), axis=0)


def to_days(df_by_day, n_cols):
    out = np.zeros((N_DAYS, n_cols), dtype="float64")
    out[df_by_day.index.values] = df_by_day.values
    return out


@lru_cache(maxsize=None)
def day_table(variant, tau=None, k=None):
    use_rel, use_chan, use_nov, hier = USES[variant]
    w = S["relevance"].values.copy() if use_rel else np.ones(len(S))
    if use_nov:
        w = w * NOV[k]
    w = w + 1e-6

    #vektor harian
    if hier:
        art_vec = group_wmean(ART, E_rep, w)                          #kalimat -> artikel
        art_w = pd.Series(w).groupby(ART).max().loc[art_vec.index].values
        art_day = pd.Series(DAY).groupby(ART).first().loc[art_vec.index].values
        day_vec = group_wmean(art_day, art_vec.values, art_w)         #artikel -> hari
    else:
        day_vec = group_wmean(DAY, E_rep, w)
    V = to_days(day_vec, E_rep.shape[1])
    has = np.zeros(N_DAYS, bool)
    has[np.unique(DAY)] = True
    tr_pos = text.index.get_indexer(tr_dates)
    tr_pos = tr_pos[has[tr_pos]]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(V[tr_pos])
    Vp = pca.transform(V)
    Vp[~has] = 0
    F = pd.DataFrame(Vp, index=text.index, columns=[f"v_{i}" for i in range(N_COMP)])

    if use_chan:
        wc = w[:, None] * channel_split(tau)
        F[[f"ch_{c}_mass" for c in CH]] = np.log1p(to_days(pd.DataFrame(wc).groupby(DAY).sum(), len(CH)))
        F[[f"ch_{c}_max" for c in CH]] = to_days(pd.DataFrame(wc).groupby(DAY).max(), len(CH))

    if use_nov:
        nv = pd.Series(NOV[k])
        F["nov_mean"] = to_days(nv.groupby(DAY).mean().to_frame(), 1)[:, 0]
        F["nov_max"] = to_days(nv.groupby(DAY).max().to_frame(), 1)[:, 0]
        art_nov = nv.groupby(ART).mean()
        art_day = pd.Series(DAY).groupby(ART).first()
        thr = art_nov[(S.groupby("art")["split"].first() == "train").values].median()
        F["nov_share_new"] = to_days((art_nov > thr).groupby(art_day.values).mean().to_frame(), 1)[:, 0]

    # --- volume
    F["has_news"] = has.astype(float)
    F["log_n_art"] = np.log1p(to_days(pd.Series(ART).groupby(DAY).nunique().to_frame(), 1)[:, 0])
    return F


COND_PREFIX = ("v_", "nov_")   #dirata-rata hanya atas hari yang punya berita, sisanya (intensitas) atas semua hari

## 7. Jendela waktu (sama untuk semua varian)

Baris tanggal t memakai hari t−19..t dengan bobot `0.5^(lag / half_life)`. Fitur "kondisional" (vektor,
novelty) dirata-rata atas hari yang ada beritanya saja; fitur intensitas (channel, volume) atas semua hari,
jadi hari tanpa berita = intensitas 0. `dk_news_frac` = porsi bobot jendela yang jatuh pada hari berberita.
20 hari pertama di `text` adalah warm-up sebelum split train, jadi baris train pertama sudah punya jendela penuh.

In [ ]:
def window_decay(F, half, window=WINDOW):
    cond = [c for c in F.columns if c.startswith(COND_PREFIX)]
    inten = [c for c in F.columns if c not in cond]
    Xc, Xi, has = F[cond].values, F[inten].values, F["has_news"].values
    n = len(F)
    num_c, den_c = np.zeros_like(Xc), np.zeros(n)
    num_i, den_i = np.zeros_like(Xi), np.zeros(n)
    for L in range(window):
        wl = 0.5 ** (L / half)
        num_c[L:] += wl * has[:n - L, None] * Xc[:n - L]
        den_c[L:] += wl * has[:n - L]
        num_i[L:] += wl * Xi[:n - L]
        den_i[L:] += wl
    out = pd.DataFrame(np.where(den_c[:, None] > 0, num_c / np.maximum(den_c, 1e-12)[:, None], 0.0),
                       index=F.index, columns=["dk_" + c for c in cond])
    out[["dk_" + c for c in inten]] = num_i / den_i[:, None]
    out["dk_news_frac"] = den_c / den_i
    return out


def attach(W):
    return {s: D[s].merge(W, left_on="date", right_index=True, how="left").fillna(0) for s in D}


def build(variant, tau, k, half):
    W = window_decay(day_table(variant, tau, k), half)
    return attach(W), list(W.columns)


def grid_for(v):
    use_rel, use_chan, use_nov, _ = USES[v]
    return list(product(GRID_TAU if use_chan else [None], GRID_K if use_nov else [None], GRID_HALF))


Dx, cols = build("hier", GRID_TAU[0], GRID_K[0], GRID_HALF[0])
print(len(cols), "fitur teks |", cols[:3], "...", cols[-4:])
assert not Dx["train"][cols].isna().any().any()

## 8. Decay + HGB / LR

Per varian: tiap kombinasi (τ, k, half-life) dinilai dengan satu setelan HGB tetap di validation
(F1-macro). Kombinasi terbaik lalu dituning penuh dan disimpan lewat `evaluate_val` untuk HGB dan LR. `n_configs_tried` = ukuran grid
varian itu. Juga dicatat interval kepercayaan 95% AUC validation (bootstrap blok 5 hari): **perbedaan antar
varian yang intervalnya tumpang tindih jauh belum tentu nyata.**

In [ ]:
#Memilih (tau, k, half) memakai SATU setelan HGB tetap (16x lebih cepat daripada tune penuh per kombinasi). setelah konfigurasi fitur terpilih, evaluate_val melakukan tuning hyperparameter penuh seperti varian C lain.
HGB_SELECT = dict(max_depth=2, learning_rate=0.03, max_iter=300, min_samples_leaf=50)

BEST = {}
t0 = time.time()
for v in VARIANTS:
    best = None
    for tau, k, half in grid_for(v):
        Dx, cols = build(v, tau, k, half)
        prob = make_model("HGB", HGB_SELECT).fit(Dx["train"][PRICE + cols], Dx["train"]["target"]) \
                                           .predict_proba(Dx["val"][PRICE + cols])[:, 1]
        f1 = metrics(Dx["val"]["target"], (prob >= 0.5).astype(int), prob)["F1_macro"]
        if best is None or f1 > best[1]:
            best = ((tau, k, half), f1)
    (tau, k, half), _ = best
    BEST[v] = dict(tau=tau, k=k, half=half)
    Dx, cols = build(v, tau, k, half)
    n_cfg = len(grid_for(v))
    note = f"C6 salience {v}: tau={tau}, k={k}, half={half}; MiniLM scorer, FinBERT sent cls, PCA{N_COMP}, window {WINDOW}"
    for kind in ["HGB", "LR"]:
        name = f"C6_{v}_{kind}"
        evaluate_val(name, kind, Dx, PRICE + cols, n_configs_tried=n_cfg, note=note)
        p = json.loads((c_common.RESULTS_C / f"{name}.json").read_text())["best_params"]
        prob = make_model(kind, p).fit(Dx["train"][PRICE + cols], Dx["train"]["target"]) \
                                  .predict_proba(Dx["val"][PRICE + cols])[:, 1]
        c_common._save(name, {"c6_config": BEST[v],
                              "val_auc_ci95": bootstrap_auc_ci(Dx["val"]["target"].values, prob)})
    print(f"  {v}: {BEST[v]} ({time.time() - t0:.0f} dtk)")
print(f"selesai dalam {time.time() - t0:.0f} dtk")
for kind in ["LR", "HGB"]:
    _, sc_price, _ = tune(kind, D, PRICE)
    print(f"REFERENSI harga saja ({kind}) val:", {a: round(b, 3) for a, b in sc_price.items()})

## 9. GRU kecil pada urutan 20 hari

Input: fitur harian (tanpa decay) hari t−19..t, distandardisasi dengan statistik hari train. Fitur harga
masuk di lapisan output. Model sengaja kecil (hidden 16, dropout) karena data train hanya ~825 hari. Early
stopping memakai loss validation; prediksi = rata-rata 5 seed. τ dan k diambil dari konfigurasi terbaik HGB
varian yang sama (half-life tidak dipakai; GRU belajar sendiri bobot waktunya).

In [ ]:
import torch.nn as nn

GRU_HIDDEN, GRU_DROP, GRU_LR, GRU_WD = 16, 0.3, 1e-3, 1e-2
GRU_EPOCHS, GRU_PATIENCE, GRU_SEEDS, GRU_BATCH = 100, 10, [0, 1, 2, 3, 4], 32


class SalienceGRU(nn.Module):
    def __init__(self, f_txt, f_price):
        super().__init__()
        self.gru = nn.GRU(f_txt, GRU_HIDDEN, batch_first=True)
        self.drop = nn.Dropout(GRU_DROP)
        self.head = nn.Linear(GRU_HIDDEN + f_price, 1)

    def forward(self, seq, price):
        _, h = self.gru(seq)
        return self.head(torch.cat([self.drop(h[-1]), price], 1)).squeeze(1)


def make_seq(variant, tau, k):
    F = day_table(variant, tau, k)
    tr_pos = text.index.get_indexer(tr_dates)
    m, s = F.values[tr_pos].mean(0), F.values[tr_pos].std(0) + 1e-8
    X = np.vstack([np.zeros((WINDOW - 1, F.shape[1])), (F.values - m) / s]).astype("float32")
    pm, ps = D["train"][PRICE].mean().values, D["train"][PRICE].std().values + 1e-8
    out = {}
    for sp in D:
        pos = text.index.get_indexer(D[sp]["date"])
        seq = np.stack([X[p:p + WINDOW] for p in pos])             
        price = ((D[sp][PRICE].values - pm) / ps).astype("float32")
        out[sp] = (torch.tensor(seq), torch.tensor(price), torch.tensor(D[sp]["target"].values, dtype=torch.float32))
    return out


def train_gru(data, tr, seed, epochs=None, va=None):
    torch.manual_seed(seed)
    seq, price, y = data[tr]
    net = SalienceGRU(seq.shape[2], price.shape[1])
    opt = torch.optim.AdamW(net.parameters(), lr=GRU_LR, weight_decay=GRU_WD)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - y.mean()) / y.mean()))
    g = torch.Generator().manual_seed(seed)
    best_loss, best_state, best_ep, bad = np.inf, None, 0, 0
    for ep in range(1, (epochs or GRU_EPOCHS) + 1):
        net.train()
        for ix in torch.randperm(len(y), generator=g).split(GRU_BATCH):
            opt.zero_grad()
            lossf(net(seq[ix], price[ix]), y[ix]).backward()
            opt.step()
        if epochs is None:
            net.eval()
            with torch.no_grad():
                vl = lossf(net(data[va][0], data[va][1]), data[va][2]).item()
            if vl < best_loss - 1e-4:
                best_loss, best_state, best_ep, bad = vl, {a: b.clone() for a, b in net.state_dict().items()}, ep, 0
            else:
                bad += 1
                if bad >= GRU_PATIENCE:
                    break
    if best_state is not None:
        net.load_state_dict(best_state)
    return net.eval(), best_ep or epochs


def predict(net, data, sp):
    with torch.no_grad():
        return torch.sigmoid(net(data[sp][0], data[sp][1])).numpy()


GRU_DATA = {}
for v in VARIANTS:
    cfg = BEST[v]
    data = make_seq(v, cfg["tau"], cfg["k"])
    GRU_DATA[v] = data
    probs, eps = [], []
    for sd_ in GRU_SEEDS:
        net, ep = train_gru(data, "train", sd_, va="val")
        probs.append(predict(net, data, "val"))
        eps.append(ep)
    prob = np.mean(probs, 0)
    y = data["val"][2].numpy().astype(int)
    sc = metrics(y, (prob >= 0.5).astype(int), prob)
    name = f"C6_{v}_GRU"
    c_common._save(name, {"name": name, "model": "GRU", "n_features": int(data["train"][0].shape[2] + len(PRICE)),
                          "best_params": dict(hidden=GRU_HIDDEN, dropout=GRU_DROP, epochs=int(round(np.mean(eps))),
                                              seeds=GRU_SEEDS, tau=cfg["tau"], k=cfg["k"]),
                          "val": sc, "n_configs_tried": len(grid_for(v)), "c6_config": cfg,
                          "val_auc_ci95": bootstrap_auc_ci(y, prob),
                          "note": f"C6 salience {v} GRU({GRU_HIDDEN}) on {WINDOW}-day sequence"})
    print(f"{name} val: " + ", ".join(f"{a}={b:.3f}" for a, b in sc.items()) + f" | epoch {eps}")

## 10. Ringkasan ablasi (validation)

In [ ]:
res = collect_results()
res = res[res["name"].str.startswith("C6_")].copy()
res["variant"] = res["name"].str.split("_").str[1]
ci = {r["name"]: json.loads((c_common.RESULTS_C / f"{r['name']}.json").read_text()).get("val_auc_ci95")
      for _, r in res.iterrows()}
res["val_auc_ci95"] = res["name"].map(ci)
order = {v: i for i, v in enumerate(VARIANTS)}
res = res.sort_values(["model", "variant"], key=lambda s: s.map(order) if s.name == "variant" else s)
show = ["name", "model", "n_features", "n_configs_tried", "val_DA", "val_F1_macro", "val_MCC", "val_AUC", "val_auc_ci95"]
display(res[show].round(3))
display(res.pivot(index="variant", columns="model", values="val_F1_macro").loc[VARIANTS].round(3))

## 11. Kontrol teks diacak & tes akhir (SEKALI)

1. Pilih **satu** varian berdasarkan tabel validation di atas, isi `CHOSEN`.
2. Jalankan kontrol "teks diacak" (khusus HGB/LR): kalau AUC asli tidak di atas kisaran acak, teks tidak benar-benar membantu.
3. Baru jalankan tes. `final_test` menolak dipanggil dua kali untuk nama yang sama.

In [ ]:
CHOSEN = None   

if CHOSEN is not None and CHOSEN[1] in ("HGB", "LR"):
    v, kind = CHOSEN
    cfg = BEST[v]
    Dx, cols = build(v, cfg["tau"], cfg["k"], cfg["half"])
    rec = json.loads((c_common.RESULTS_C / f"C6_{v}_{kind}.json").read_text())
    print("kontrol teks diacak:", shuffled_text_control(kind, Dx, PRICE, cols, rec["best_params"]))
    print("AUC val asli:", round(rec["val"]["AUC"], 3))

In [ ]:
def final_test_gru(v):
    name = f"C6_{v}_GRU"
    rec = json.loads((c_common.RESULTS_C / f"{name}.json").read_text())
    assert "test" not in rec, f"{name} sudah pernah dites; jangan pilih ulang berdasarkan test."
    data = GRU_DATA[v]
    trval = {"trval": tuple(torch.cat([data["train"][i], data["val"][i]]) for i in range(3)), "test": data["test"]}
    probs = [predict(train_gru(trval, "trval", s, epochs=rec["best_params"]["epochs"])[0], trval, "test")
             for s in GRU_SEEDS]
    prob = np.mean(probs, 0)
    y = data["test"][2].numpy().astype(int)
    sc = metrics(y, (prob >= 0.5).astype(int), prob)
    ci = bootstrap_auc_ci(y, prob)
    c_common._save(name, {"test": sc, "test_auc_ci95": ci})
    print(f"{name} TEST: " + ", ".join(f"{a}={b:.3f}" for a, b in sc.items()) + f" | AUC 95% CI {ci}")
    return prob


RUN_TEST = False     #ubah ke True hanya sekali, setelah CHOSEN dipilih
if RUN_TEST and CHOSEN is not None:
    v, kind = CHOSEN
    if kind == "GRU":
        final_test_gru(v)
    else:
        cfg = BEST[v]
        Dx, cols = build(v, cfg["tau"], cfg["k"], cfg["half"])
        final_test(f"C6_{v}_{kind}", kind, Dx, PRICE + cols)